In [1]:
from stock_embeddings import ReturnsData
from stock_embeddings.context.window import get_target_context_sets, Euclidean, ICCBRMetric

# Constants and configurations
TRAIN_PCT = 1
CONTEXT_SIZE = 32
GRANULARITY = 1 # 1=daily, 5=weekly, etc.
WINDOW_LENGTHS = [20]
STRIDE = None

data = ReturnsData()

data = ReturnsData()
data.change_returns_period(GRANULARITY)
data.train_test_split(TRAIN_PCT)
    
tgt_context_sets = []
for window_length in WINDOW_LENGTHS:
    print(f"Running for period {window_length}")
    
    tgt_context_sets += get_target_context_sets(
        X=data.returns_df.select(data.tickers).to_numpy().T,
        metric_class=Euclidean(),
        window_length=window_length,
        stride=STRIDE,
        context_size=32,
        verbose=True,
    )
    print("="*20)
print(f"Total number of target context sets: {len(tgt_context_sets)}")

No change made because period entered is 1
Running for period 20


  0%|          | 0/234 [00:00<?, ?it/s]

 10%|▉         | 23/234 [00:00<00:00, 224.37it/s]

 20%|█▉        | 46/234 [00:00<00:01, 183.02it/s]

 30%|██▉       | 70/234 [00:00<00:00, 202.71it/s]

...

 70%|███████   | 164/234 [00:00<00:00, 212.42it/s]

 80%|████████  | 188/234 [00:00<00:00, 218.16it/s]

 90%|█████████ | 211/234 [00:01<00:00, 190.58it/s]

100%|██████████| 234/234 [00:01<00:00, 202.65it/s]

Total number of target context sets: 142974


In [2]:
import polars as pl
import numpy as np

ticker = "JPM"
i = data.ticker2idx[ticker]
temp = np.array([xi[1] for xi in tgt_context_sets if xi[0] == i]).flatten()
top_values = pl.Series("value", temp).value_counts(sort=True)["value"].to_list()
print(
    f"The most commonly co-occurring stocks with {ticker} are: {[data.idx2ticker[xi] for xi in top_values][:5]}"
)

The most commonly co-occurring stocks with JPM are: ['C', 'WFC', 'PNC', 'USB', 'BAC']


In [3]:
from stock_embeddings import ClassificationEmbeddings

EMBEDDING_DIM = 20
model = ClassificationEmbeddings(
    n_time_series=len(data.tickers), embedding_dim=EMBEDDING_DIM
)

In [4]:
from stock_embeddings import train_embeddings_from_tgt_context_sets


EPOCHS = 10
model, losses = train_embeddings_from_tgt_context_sets(
    n_time_series=len(data.tickers),
    tgt_context_sets=tgt_context_sets,
    model=model,
    epochs=EPOCHS,
    batch_size=64,
    early_stopping=False,
    device="cpu",
    # embedding_dim=EMBEDDING_DIM,
    verbose=True,
)

Training embeddings...


  0%|          | 0/10 [00:00<?, ?it/s]

 10%|█         | 1/10 [00:01<00:12,  1.37s/it]

Epoch 0: Loss = 0.10534136293583582


 20%|██        | 2/10 [00:02<00:10,  1.35s/it]

...

 90%|█████████ | 9/10 [00:12<00:01,  1.35s/it]

Epoch 8: Loss = 0.0996309985067617


100%|██████████| 10/10 [00:13<00:00,  1.35s/it]

100%|██████████| 10/10 [00:13<00:00,  1.35s/it]

Epoch 9: Loss = 0.09886093187761905


In [5]:
from stock_embeddings import get_sector_score

get_sector_score(model.embeddings.weight.detach().numpy(), sectors=data.sectors)

/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i

Precision Score: 0.26
Recall Score: 0.25
F1 Score: 0.25
Accuracy Score: 0.25


/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [6]:
SAVE_MODEL = False
SAVE_PATH_TEMPLATE = "embeddings/abs_diff_E{epochs}_C{context_size}_D{embedding_dim}_P{periods}_train{train_pct}.pt"